In [ ]:
# Clone the ultralytics repository
!git clone https://github.com/ultralytics/ultralytics


fatal: destination path 'ultralytics' already exists and is not an empty directory.


In [ ]:
# Navigate to the directory containing requirements.txt
%cd /content/ultralytics/examples/YOLOv8-Action-Recognition/

# Install the required packages
!pip install -r requirements.txt

/content/ultralytics/examples/YOLOv8-Action-Recognition


In [ ]:
%cd /content/ultralytics

/content/ultralytics


In [ ]:
import base64, io
from PIL import Image
import cv2, numpy as np
from ultralytics import YOLO
from google.colab import output

model = YOLO('yolov8n.pt')
stop_flag = False

def handleFrameJS(b64image):
    global stop_flag
    if stop_flag:
        return None
    decoded = base64.b64decode(b64image.split(',')[1])
    img = Image.open(io.BytesIO(decoded))
    frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    results = model(frame, conf=0.3)[0]
    boxes = results.xyxy[0].cpu().numpy().tolist()
    return {'boxes': boxes, 'names': model.names}

def stopLiveFeedJS():
    global stop_flag
    stop_flag = True

def resetLiveFeedJS():
    global stop_flag
    stop_flag = False

output.register_callback('notebook.handleFrameJS', handleFrameJS)
output.register_callback('notebook.stopLiveFeedJS', stopLiveFeedJS)
output.register_callback('notebook.resetLiveFeedJS', resetLiveFeedJS)


In [ ]:
%%html
<!DOCTYPE html>
<html>
<head>
  <style>
    body {
      margin: 0; padding: 20px;
      font-family: Arial, sans-serif;
      background: #f7f8fa;
      text-align: center;
    }
    h2 { color: #333; }
    #controls { margin: 15px 0; }
    button {
      margin: 0 8px; padding: 10px 18px;
      font-size: 16px; border: none; border-radius: 4px;
      cursor: pointer; box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    }
    #startBtn { background: #28a745; color: white; }
    #stopBtn  { background: #dc3545; color: white; }
    canvas {
      border: 2px solid #444; border-radius: 4px;
      max-width: 100%; height: auto;
    }
  </style>
</head>
<body>
  <h2> Real Time Object Detection</h2>
  <div id="controls">
    <button id="startBtn">▶️ Start Camera</button>
    <button id="stopBtn" disabled>🛑 Stop Camera</button>
  </div>
  <canvas id="canvas"></canvas>

  <script>
    let video, stream, intervalId;
    let jsRunning = false;
    const canvas = document.getElementById('canvas');
    const ctx    = canvas.getContext('2d');
    const startBtn = document.getElementById('startBtn');
    const stopBtn  = document.getElementById('stopBtn');

    startBtn.onclick = async () => {
      // Reset Python stop_flag
      await google.colab.kernel.invokeFunction('notebook.resetLiveFeedJS', [], {});

      jsRunning = true;
      startBtn.disabled = true;
      stopBtn.disabled  = false;

      // Cleanup any prior run
      if (intervalId) clearInterval(intervalId);
      if (video) {
        video.srcObject.getTracks().forEach(t=>t.stop());
        video.remove();
      }

      // Hidden video element
      video = document.createElement('video');
      video.style.display = 'none';
      video.setAttribute('playsinline', '');
      document.body.appendChild(video);

      stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      await video.play();

      // Resize canvas
      canvas.width  = video.videoWidth;
      canvas.height = video.videoHeight;

      // Frame loop (~10 FPS)
      intervalId = setInterval(async () => {
        if (!jsRunning) return;
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        const jpeg = canvas.toDataURL('image/jpeg', 0.8);
        const res = await google.colab.kernel.invokeFunction(
          'notebook.handleFrameJS', [jpeg], {}
        );
        if (!res || !res.boxes) return;

        // Draw boxes & labels
        ctx.lineWidth = 2;
        ctx.font = '16px Arial';
        ctx.fillStyle = 'lime';
        ctx.strokeStyle = 'lime';
        res.boxes.forEach(b => {
          const [x1,y1,x2,y2,conf,cls] = b;
          const name = res.names[cls];
          ctx.beginPath();
          ctx.rect(x1, y1, x2 - x1, y2 - y1);
          ctx.stroke();
          ctx.fillText(`${name} ${(conf*100).toFixed(1)}%`, x1, y1>20?y1-5:y1+15);
        });
      }, 100);
    };

    stopBtn.onclick = () => {
      jsRunning = false;
      startBtn.disabled = false;
      stopBtn.disabled  = true;
      if (intervalId) clearInterval(intervalId);
      if (stream) stream.getTracks().forEach(t=>t.stop());
      if (video) video.remove();
      // Tell Python to stop too
      google.colab.kernel.invokeFunction('notebook.stopLiveFeedJS', [], {});
    };
  </script>
</body>
</html>



0: 480x640 1 person, 221.9ms
Speed: 6.5ms preprocess, 221.9ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 224.5ms
Speed: 4.0ms preprocess, 224.5ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 216.2ms
Speed: 5.4ms preprocess, 216.2ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 223.1ms
Speed: 4.9ms preprocess, 223.1ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 214.7ms
Speed: 4.6ms preprocess, 214.7ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 223.7ms
Speed: 4.6ms preprocess, 223.7ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 209.3ms
Speed: 4.8ms preprocess, 209.3ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 218.8ms
Speed: 4.5ms preprocess, 218.8ms inference, 1.9ms postprocess per image at